In [4]:
from google.colab import files

uploaded = files.upload()

Saving fitroutine1.csv to fitroutine1.csv


In [6]:

import pandas as pd
import numpy as np

# Load the recreated FitRoutine dataset
df = pd.read_csv("fitroutine1.csv")

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())


Dataset loaded successfully!
Rows: 544509
Columns: 16


,title,description,level,goal,equipment,program_length,time_per_workout,week,day,number_of_exercises,exercise_name,sets,reps,intensity,created,last_edit
0,Lyle McDonald Routine (Strength/Hypertrophy Vers),IMPORTANT: THIS IS NOT THE ORIGINAL LYLE MCDON...,[],[],Full Gym,12.0,90.0,1.0,1.0,6,Bench Press (Barbell),5.0,9.0,8.0,2025-05-16 11:23:00,2025-06-27 02:28:00
1,Lyle McDonald Routine (Strength/Hypertrophy Vers),IMPORTANT: THIS IS NOT THE ORIGINAL LYLE MCDON...,[],[],Full Gym,12.0,90.0,1.0,1.0,6,Bent Over Row (Barbell),5.0,9.0,8.0,2025-05-16 11:23:00,2025-06-27 02:28:00
2,Lyle McDonald Routine (Strength/Hypertrophy Vers),IMPORTANT: THIS IS NOT THE ORIGINAL LYLE MCDON...,[],[],Full Gym,12.0,90.0,1.0,1.0,6,Shoulder Press (Machine),3.0,14.0,8.0,2025-05-16 11:23:00,2025-06-27 02:28:00
3,Lyle McDonald Routine (Strength/Hypertrophy Vers),IMPORTANT: THIS IS NOT THE ORIGINAL LYLE MCDON...,"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Full Gym,12.0,90.0,1.0,1.0,6,Lat Pulldown,3.0,14.0,8.0,2025-05-16 11:23:00,2025-06-27 02:28:00
4,Lyle McDonald Routine (Strength/Hypertrophy Vers),IMPORTANT: THIS IS NOT THE ORIGINAL LYLE MCDON...,[],[],Full Gym,12.0,90.0,1.0,1.0,6,Tricep Extension (Cable),2.0,5.0,8.0,2025-05-16 11:23:00,2025-06-27 02:28:00


In [7]:

import ast
import pandas as pd

# Convert the level column from text into lists
def parse_levels(value):
    if pd.isna(value):
        return []

    try:
        result = ast.literal_eval(str(value))
        return result if isinstance(result, list) else []
    except (ValueError, SyntaxError):
        return []

df["parsed_level"] = df["level"].apply(parse_levels)

# Combine difficulty labels across all rows of each program
program_levels = (
    df.groupby("title")["parsed_level"]
    .agg(lambda rows: sorted(set(
        level
        for row in rows
        for level in row
    )))
    .reset_index(name="levels")
)

# Count difficulty labels per program
program_levels["level_count"] = (
    program_levels["levels"].apply(len)
)

print("=== FITROUTINE LEVEL ANALYSIS ===")
print("Total programs:", len(program_levels))

print("\nNumber of levels per program:")
print(
    program_levels["level_count"]
    .value_counts()
    .sort_index()
)

print("\nFrequency of each difficulty level:")
print(
    program_levels["levels"]
    .explode()
    .dropna()
    .value_counts()
)

print("\nExample programs:")
display(program_levels.head(10))


=== FITROUTINE LEVEL ANALYSIS ===
Total programs: 2537

Number of levels per program:
level_count
0      2
1    461
2    806
3    707
4    561
Name: count, dtype: int64

Frequency of each difficulty level:
levels
Intermediate    2228
Novice          1754
Advanced        1258
Beginner        1198
Name: count, dtype: int64

Example programs:


,title,levels,level_count
0,(MASS MONSTER) High Intensity 4 Day Upper Lowe...,[Intermediate],1
1,(NOT MY PROGRAM)SHJ Jotaro,"[Advanced, Intermediate]",2
2,1 PowerLift Per Day Powerbuilding 5 Day Bro Split,"[Advanced, Beginner, Intermediate, Novice]",4
3,10 Week Mass Building Program,"[Advanced, Intermediate, Novice]",3
4,10 week deadlift focus,"[Advanced, Intermediate]",2
5,1000 lbs Club,"[Intermediate, Novice]",2
6,109273941442,"[Beginner, Novice]",2
7,10x3 Powerbuilding,"[Advanced, Beginner, Intermediate, Novice]",4
8,12 WEEK FAT DESTROYER.,"[Advanced, Beginner, Intermediate, Novice]",4
9,12 Week Amped Up,"[Beginner, Intermediate, Novice]",3


In [8]:

# Analyze equipment labels at the program level

equipment_data = df[["title", "equipment"]].copy()

equipment_data["equipment"] = (
    equipment_data["equipment"]
    .fillna("Missing")
    .astype(str)
    .str.strip()
)

# Check whether each program has one equipment category
program_equipment = (
    equipment_data.drop_duplicates()
)

print("=== FITROUTINE EQUIPMENT ANALYSIS ===")

print("\nTotal programs:",
      program_equipment["title"].nunique())

print("\nEquipment categories:")
print(
    program_equipment["equipment"]
    .value_counts(dropna=False)
)

print("\nPrograms with multiple equipment categories:")
print(
    program_equipment.groupby("title")["equipment"]
    .nunique()
    .gt(1)
    .sum()
)

print("\nExample records:")
display(program_equipment.head(10))


=== FITROUTINE EQUIPMENT ANALYSIS ===

Total programs: 2537

Equipment categories:
equipment
Full Gym         1838
Garage Gym        531
At Home           108
Dumbbell Only      68
Missing             1
Name: count, dtype: int64

Programs with multiple equipment categories:
9

Example records:


,title,equipment
0,Lyle McDonald Routine (Strength/Hypertrophy Vers),Full Gym
288,"Jeff Nippard’s Fundamentals, Upper&Lower Split...",Full Gym
512,nSuns_Rebuild by KSI,Full Gym
652,Rampage,Full Gym
1210,Fitness King Powerbuild 101,Full Gym
1527,Nippard’s The Bodybuilding Transformation System,Full Gym
1897,Strong Curves: Gluteal Goddess,Full Gym
2057,My Upper/Lower Split,Full Gym
2243,M&S: 4 Day Upper/Lower Dumbbell Workout Routine,Dumbbell Only
2555,Larry's Modified Bullmastiff Powerbuilding Pro...,Full Gym


In [9]:

# STEP 5: Prepare program-level equipment labels

# One row per unique program-equipment combination
equipment_labels = (
    df[["title", "equipment"]]
    .drop_duplicates()
    .copy()
)

equipment_labels["equipment"] = (
    equipment_labels["equipment"]
    .astype("string")
    .str.strip()
)

# Remove missing or empty equipment labels
equipment_labels = equipment_labels[
    equipment_labels["equipment"].notna()
    & equipment_labels["equipment"].ne("")
]

# Identify programs with exactly one equipment category
label_counts = (
    equipment_labels.groupby("title")["equipment"]
    .nunique()
)

valid_titles = label_counts[
    label_counts == 1
].index

clean_equipment = (
    equipment_labels[
        equipment_labels["title"].isin(valid_titles)
    ]
    .drop_duplicates(subset=["title"])
    .reset_index(drop=True)
)

print("=== CLEAN EQUIPMENT DATASET ===")
print("Programs retained:", len(clean_equipment))
print("Programs excluded:",
      df["title"].nunique() - len(clean_equipment))

print("\nEquipment distribution:")
print(clean_equipment["equipment"].value_counts())

display(clean_equipment.head())


=== CLEAN EQUIPMENT DATASET ===
Programs retained: 2527
Programs excluded: 10

Equipment distribution:
equipment
Full Gym         1830
Garage Gym        523
At Home           107
Dumbbell Only      67
Name: count, dtype: Int64


,title,equipment
0,Lyle McDonald Routine (Strength/Hypertrophy Vers),Full Gym
1,"Jeff Nippard’s Fundamentals, Upper&Lower Split...",Full Gym
2,nSuns_Rebuild by KSI,Full Gym
3,Rampage,Full Gym
4,Fitness King Powerbuild 101,Full Gym


In [10]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

# Collect unique exercises for each program
program_exercises = (
    df.groupby("title")["exercise_name"]
    .agg(lambda x: sorted(set(x.dropna())))
    .reset_index(name="exercises")
)

# Combine exercises with clean equipment labels
model_data = clean_equipment.merge(
    program_exercises,
    on="title",
    how="inner",
    validate="one_to_one"
)

# Split by program before encoding exercises
train_data, val_data = train_test_split(
    model_data,
    test_size=0.20,
    random_state=42,
    stratify=model_data["equipment"]
)

# Fit exercise encoder on training programs only
exercise_encoder = MultiLabelBinarizer(
    sparse_output=True
)

X_train = exercise_encoder.fit_transform(
    train_data["exercises"]
)

X_val = exercise_encoder.transform(
    val_data["exercises"]
)

y_train = train_data["equipment"].to_numpy()
y_val = val_data["equipment"].to_numpy()

print("=== TRAINING AND VALIDATION DATA ===")
print("Training programs:", len(train_data))
print("Validation programs:", len(val_data))

print("\nTraining feature shape:", X_train.shape)
print("Validation feature shape:", X_val.shape)

print("\nTraining equipment distribution:")
print(train_data["equipment"].value_counts())

print("\nValidation equipment distribution:")
print(val_data["equipment"].value_counts())

print("\nShared program titles:",
      len(set(train_data["title"]) & set(val_data["title"])))


=== TRAINING AND VALIDATION DATA ===
Training programs: 2021
Validation programs: 506

Training feature shape: (2021, 300)
Validation feature shape: (506, 300)

Training equipment distribution:
equipment
Full Gym         1463
Garage Gym        418
At Home            86
Dumbbell Only      54
Name: count, dtype: Int64

Validation equipment distribution:
equipment
Full Gym         367
Garage Gym       105
At Home           21
Dumbbell Only     13
Name: count, dtype: Int64

Shared program titles: 0


In [11]:

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)
import pandas as pd

# Train Decision Tree
dt_model = DecisionTreeClassifier(
    max_depth=8,
    min_samples_leaf=10,
    random_state=42
)

dt_model.fit(X_train, y_train)

# Make predictions
train_pred = dt_model.predict(X_train)
val_pred = dt_model.predict(X_val)

# Calculate accuracy
train_accuracy = accuracy_score(y_train, train_pred)
val_accuracy = accuracy_score(y_val, val_pred)

# Calculate generalization gap
accuracy_gap = abs(
    train_accuracy - val_accuracy
) * 100

# Calculate balanced accuracy
balanced_acc = balanced_accuracy_score(
    y_val, val_pred
)

print("=== DECISION TREE RESULTS ===")

print(f"Training accuracy:   {train_accuracy * 100:.2f}%")
print(f"Validation accuracy: {val_accuracy * 100:.2f}%")
print(f"Accuracy gap:        {accuracy_gap:.2f} percentage points")
print(f"Balanced accuracy:   {balanced_acc * 100:.2f}%")

print("\n=== CLASSIFICATION REPORT ===")
print(
    classification_report(
        y_val,
        val_pred,
        zero_division=0
    )
)

print("\n=== REQUIREMENTS ===")
print(
    "Validation accuracy >= 85%:",
    "PASS" if val_accuracy >= 0.85 else "NOT YET"
)

print(
    "Accuracy gap <= 5 percentage points:",
    "PASS" if accuracy_gap <= 5 else "NOT YET"
)


=== DECISION TREE RESULTS ===
Training accuracy:   77.29%
Validation accuracy: 72.33%
Accuracy gap:        4.96 percentage points
Balanced accuracy:   51.16%

=== CLASSIFICATION REPORT ===
               precision    recall  f1-score   support

      At Home       0.52      0.52      0.52        21
Dumbbell Only       0.50      0.31      0.38        13
     Full Gym       0.81      0.85      0.83       367
   Garage Gym       0.41      0.36      0.39       105

     accuracy                           0.72       506
    macro avg       0.56      0.51      0.53       506
 weighted avg       0.71      0.72      0.72       506


=== REQUIREMENTS ===
Validation accuracy >= 85%: NOT YET
Accuracy gap <= 5 percentage points: PASS


In [12]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report
)

# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=3,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=1
)

rf_model.fit(X_train, y_train)

# Predictions
rf_train_pred = rf_model.predict(X_train)
rf_val_pred = rf_model.predict(X_val)

# Evaluation
rf_train_acc = accuracy_score(y_train, rf_train_pred)
rf_val_acc = accuracy_score(y_val, rf_val_pred)

rf_gap = abs(rf_train_acc - rf_val_acc) * 100

rf_balanced_acc = balanced_accuracy_score(
    y_val, rf_val_pred
)

print("=== RANDOM FOREST EQUIPMENT RESULTS ===")
print(f"Training accuracy:   {rf_train_acc * 100:.2f}%")
print(f"Validation accuracy: {rf_val_acc * 100:.2f}%")
print(f"Accuracy gap:        {rf_gap:.2f} percentage points")
print(f"Balanced accuracy:   {rf_balanced_acc * 100:.2f}%")

print("\n=== CLASSIFICATION REPORT ===")
print(
    classification_report(
        y_val,
        rf_val_pred,
        zero_division=0
    )
)

print("\n=== REQUIREMENTS ===")
print(
    "Validation accuracy >= 85%:",
    "PASS" if rf_val_acc >= 0.85 else "NOT YET"
)

print(
    "Accuracy gap <= 5 percentage points:",
    "PASS" if rf_gap <= 5 else "NOT YET"
)


=== RANDOM FOREST EQUIPMENT RESULTS ===
Training accuracy:   83.77%
Validation accuracy: 71.94%
Accuracy gap:        11.83 percentage points
Balanced accuracy:   64.82%

=== CLASSIFICATION REPORT ===
               precision    recall  f1-score   support

      At Home       0.35      0.57      0.44        21
Dumbbell Only       0.45      0.69      0.55        13
     Full Gym       0.89      0.78      0.83       367
   Garage Gym       0.45      0.55      0.49       105

     accuracy                           0.72       506
    macro avg       0.53      0.65      0.58       506
 weighted avg       0.76      0.72      0.73       506


=== REQUIREMENTS ===
Validation accuracy >= 85%: NOT YET
Accuracy gap <= 5 percentage points: NOT YET


In [13]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report
)

# Train Logistic Regression
lr_model = LogisticRegression(
    max_iter=2000,
    C=0.1,
    solver="lbfgs",
    random_state=42
)

lr_model.fit(X_train, y_train)

# Predictions
lr_train_pred = lr_model.predict(X_train)
lr_val_pred = lr_model.predict(X_val)

# Evaluation
lr_train_acc = accuracy_score(y_train, lr_train_pred)
lr_val_acc = accuracy_score(y_val, lr_val_pred)

lr_gap = abs(lr_train_acc - lr_val_acc) * 100

lr_balanced_acc = balanced_accuracy_score(
    y_val, lr_val_pred
)

print("=== LOGISTIC REGRESSION EQUIPMENT RESULTS ===")
print(f"Training accuracy:   {lr_train_acc * 100:.2f}%")
print(f"Validation accuracy: {lr_val_acc * 100:.2f}%")
print(f"Accuracy gap:        {lr_gap:.2f} percentage points")
print(f"Balanced accuracy:   {lr_balanced_acc * 100:.2f}%")

print("\n=== CLASSIFICATION REPORT ===")
print(
    classification_report(
        y_val,
        lr_val_pred,
        zero_division=0
    )
)

print("\n=== REQUIREMENTS ===")
print(
    "Validation accuracy >= 85%:",
    "PASS" if lr_val_acc >= 0.85 else "NOT YET"
)

print(
    "Accuracy gap <= 5 percentage points:",
    "PASS" if lr_gap <= 5 else "NOT YET"
)


=== LOGISTIC REGRESSION EQUIPMENT RESULTS ===
Training accuracy:   84.02%
Validation accuracy: 79.25%
Accuracy gap:        4.77 percentage points
Balanced accuracy:   56.31%

=== CLASSIFICATION REPORT ===
               precision    recall  f1-score   support

      At Home       0.71      0.48      0.57        21
Dumbbell Only       0.45      0.38      0.42        13
     Full Gym       0.85      0.92      0.88       367
   Garage Gym       0.57      0.48      0.52       105

     accuracy                           0.79       506
    macro avg       0.65      0.56      0.60       506
 weighted avg       0.78      0.79      0.78       506


=== REQUIREMENTS ===
Validation accuracy >= 85%: NOT YET
Accuracy gap <= 5 percentage points: PASS


In [14]:

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score

results = []

for c_value in [0.01, 0.03, 0.1, 0.3, 1.0, 3.0]:

    model = LogisticRegression(
        C=c_value,
        max_iter=2000,
        solver="lbfgs",
        random_state=42
    )

    model.fit(X_train, y_train)

    train_predictions = model.predict(X_train)
    val_predictions = model.predict(X_val)

    train_acc = accuracy_score(
        y_train, train_predictions
    ) * 100

    val_acc = accuracy_score(
        y_val, val_predictions
    ) * 100

    gap = abs(train_acc - val_acc)

    balanced_acc = balanced_accuracy_score(
        y_val, val_predictions
    ) * 100

    results.append({
        "C": c_value,
        "Training accuracy (%)": round(train_acc, 2),
        "Validation accuracy (%)": round(val_acc, 2),
        "Gap (points)": round(gap, 2),
        "Balanced accuracy (%)": round(balanced_acc, 2),
        "Meets both requirements": (
            val_acc >= 85 and gap <= 5
        )
    })

results_df = pd.DataFrame(results)

display(results_df)


,C,Training accuracy (%),Validation accuracy (%),Gap (points),Balanced accuracy (%),Meets both requirements
0,0.01,76.50,75.30,1.20,32.11,False
1,0.03,81.00,78.85,2.15,51.89,False
2,0.10,84.02,79.25,4.77,56.31,False
3,0.30,85.85,77.47,8.38,60.09,False
4,1.00,86.89,76.88,10.01,59.10,False
5,3.00,88.08,75.30,12.78,63.22,False


In [15]:

import pandas as pd
from sklearn.metrics import confusion_matrix

# Use the existing Logistic Regression model with C=0.1
labels = [
    "At Home",
    "Dumbbell Only",
    "Full Gym",
    "Garage Gym"
]

cm = confusion_matrix(
    y_val,
    lr_val_pred,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=[f"Actual: {x}" for x in labels],
    columns=[f"Predicted: {x}" for x in labels]
)

print("=== LOGISTIC REGRESSION CONFUSION MATRIX ===")
display(cm_df)

print("\n=== FULL GYM / GARAGE GYM CONFUSION ===")

print(
    "Actual Garage Gym predicted as Full Gym:",
    cm_df.loc[
        "Actual: Garage Gym",
        "Predicted: Full Gym"
    ]
)

print(
    "Actual Full Gym predicted as Garage Gym:",
    cm_df.loc[
        "Actual: Full Gym",
        "Predicted: Garage Gym"
    ]
)


=== LOGISTIC REGRESSION CONFUSION MATRIX ===


,Predicted: At Home,Predicted: Dumbbell Only,Predicted: Full Gym,Predicted: Garage Gym
Actual: At Home,10,3,5,3
Actual: Dumbbell Only,3,5,2,3
Actual: Full Gym,0,0,336,31
Actual: Garage Gym,1,3,51,50



=== FULL GYM / GARAGE GYM CONFUSION ===
Actual Garage Gym predicted as Full Gym: 51
Actual Full Gym predicted as Garage Gym: 31


In [16]:

import pandas as pd

# Use training programs only for this analysis
training_exercises = train_data[
    ["title", "equipment", "exercises"]
].explode("exercises")

# Count the number of programs containing each exercise
exercise_counts = (
    training_exercises.groupby(
        ["equipment", "exercises"]
    )["title"]
    .nunique()
    .reset_index(name="program_count")
)

# Calculate proportion within each equipment category
category_sizes = (
    train_data["equipment"]
    .value_counts()
)

exercise_counts["percentage"] = (
    exercise_counts["program_count"]
    / exercise_counts["equipment"].map(category_sizes)
    * 100
)

for category in ["Full Gym", "Garage Gym"]:
    print(f"\n=== TOP EXERCISES: {category} ===")

    top_exercises = (
        exercise_counts[
            exercise_counts["equipment"] == category
        ]
        .sort_values(
            "program_count",
            ascending=False
        )
        .head(15)
    )

    display(
        top_exercises[
            ["exercises", "program_count", "percentage"]
        ]
    )



=== TOP EXERCISES: Full Gym ===


,exercises,program_count,percentage
530,Leg Extension,950,64.935065
412,Bench Press (Barbell),813,55.570745
653,Squat (Barbell),796,54.408749
522,Lat Pulldown,780,53.315106
527,Lateral Raise (Dumbbell),731,49.965824
506,Incline Bench Press (Dumbbell),657,44.907724
605,Romanian Deadlift (Barbell),600,41.01162
452,Deadlift (Barbell),557,38.072454
619,Seated Row (Cable),543,37.115516
529,Leg Curl,505,34.518113



=== TOP EXERCISES: Garage Gym ===


,exercises,program_count,percentage
712,Bench Press (Barbell),259,61.961722
951,Squat (Barbell),250,59.808612
752,Deadlift (Barbell),202,48.325359
827,Lateral Raise (Dumbbell),187,44.736842
904,Romanian Deadlift (Barbell),156,37.320574
855,Overhead Press (Barbell),145,34.688995
806,Incline Bench Press (Dumbbell),129,30.861244
958,Standing Calf Raise,120,28.708134
830,Leg Extension,119,28.4689
822,Lat Pulldown,113,27.033493


In [17]:

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report
)

# Count how often each exercise appears in each program
exercise_frequency = pd.crosstab(
    df["title"],
    df["exercise_name"]
)

# Keep the same 300 exercise columns
exercise_frequency = exercise_frequency.reindex(
    columns=exercise_encoder.classes_,
    fill_value=0
)

# Align with our existing training and validation split
X_train_freq = exercise_frequency.reindex(
    train_data["title"],
    fill_value=0
)

X_val_freq = exercise_frequency.reindex(
    val_data["title"],
    fill_value=0
)

# Normalize counts within each program
# so longer programs don't automatically have larger values
X_train_freq = X_train_freq.div(
    X_train_freq.sum(axis=1).replace(0, 1),
    axis=0
)

X_val_freq = X_val_freq.div(
    X_val_freq.sum(axis=1).replace(0, 1),
    axis=0
)

# Train Logistic Regression with frequency features
lr_freq = LogisticRegression(
    C=0.1,
    max_iter=2000,
    solver="lbfgs",
    random_state=42
)

lr_freq.fit(X_train_freq, y_train)

# Predictions
freq_train_pred = lr_freq.predict(X_train_freq)
freq_val_pred = lr_freq.predict(X_val_freq)

# Metrics
freq_train_acc = accuracy_score(
    y_train, freq_train_pred
) * 100

freq_val_acc = accuracy_score(
    y_val, freq_val_pred
) * 100

freq_gap = abs(freq_train_acc - freq_val_acc)

freq_balanced = balanced_accuracy_score(
    y_val, freq_val_pred
) * 100

print("=== FREQUENCY-BASED LOGISTIC REGRESSION ===")
print(f"Training accuracy:   {freq_train_acc:.2f}%")
print(f"Validation accuracy: {freq_val_acc:.2f}%")
print(f"Accuracy gap:        {freq_gap:.2f} percentage points")
print(f"Balanced accuracy:   {freq_balanced:.2f}%")

print("\n=== CLASSIFICATION REPORT ===")
print(
    classification_report(
        y_val,
        freq_val_pred,
        zero_division=0
    )
)

print("\n=== REQUIREMENTS ===")
print("Validation accuracy >= 85%:",
      "PASS" if freq_val_acc >= 85 else "NOT YET")

print("Gap <= 5 percentage points:",
      "PASS" if freq_gap <= 5 else "NOT YET")


=== FREQUENCY-BASED LOGISTIC REGRESSION ===
Training accuracy:   72.39%
Validation accuracy: 72.53%
Accuracy gap:        0.14 percentage points
Balanced accuracy:   25.00%

=== CLASSIFICATION REPORT ===
               precision    recall  f1-score   support

      At Home       0.00      0.00      0.00        21
Dumbbell Only       0.00      0.00      0.00        13
     Full Gym       0.73      1.00      0.84       367
   Garage Gym       0.00      0.00      0.00       105

     accuracy                           0.73       506
    macro avg       0.18      0.25      0.21       506
 weighted avg       0.53      0.73      0.61       506


=== REQUIREMENTS ===
Validation accuracy >= 85%: NOT YET
Gap <= 5 percentage points: PASS


In [18]:

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report
)

# Create program-level workout structure features
workout_stats = (
    df.groupby("title")
    .agg(
        total_exercise_entries=("exercise_name", "count"),
        unique_exercises=("exercise_name", "nunique"),
        average_sets=("sets", "mean"),
        average_program_length=("program_length", "mean")
    )
)

# Count unique workout days using week + day combinations
workout_days = (
    df[["title", "week", "day"]]
    .drop_duplicates()
    .groupby("title")
    .size()
)

workout_stats["total_workout_days"] = workout_days

workout_stats["exercises_per_day"] = (
    workout_stats["total_exercise_entries"]
    / workout_stats["total_workout_days"].replace(0, np.nan)
)

# Align features with the existing program split
stats_train = workout_stats.reindex(train_data["title"])
stats_val = workout_stats.reindex(val_data["title"])

# Handle missing and non-finite values using training statistics
stats_train = stats_train.replace([np.inf, -np.inf], np.nan)
stats_val = stats_val.replace([np.inf, -np.inf], np.nan)

train_medians = stats_train.median().fillna(0)

stats_train = stats_train.fillna(train_medians)
stats_val = stats_val.fillna(train_medians)

# Standardize using training data only
scaler = StandardScaler()

stats_train_scaled = scaler.fit_transform(stats_train)
stats_val_scaled = scaler.transform(stats_val)

# Combine original binary exercise features with workout statistics
X_train_combined = hstack([
    X_train,
    csr_matrix(stats_train_scaled)
]).tocsr()

X_val_combined = hstack([
    X_val,
    csr_matrix(stats_val_scaled)
]).tocsr()

# Train Logistic Regression
lr_combined = LogisticRegression(
    C=0.1,
    max_iter=2000,
    solver="lbfgs",
    random_state=42
)

lr_combined.fit(X_train_combined, y_train)

combined_train_pred = lr_combined.predict(X_train_combined)
combined_val_pred = lr_combined.predict(X_val_combined)

train_acc = accuracy_score(y_train, combined_train_pred) * 100
val_acc = accuracy_score(y_val, combined_val_pred) * 100
gap = abs(train_acc - val_acc)

balanced_acc = balanced_accuracy_score(
    y_val, combined_val_pred
) * 100

print("=== COMBINED FEATURES LOGISTIC REGRESSION ===")
print(f"Training accuracy:   {train_acc:.2f}%")
print(f"Validation accuracy: {val_acc:.2f}%")
print(f"Accuracy gap:        {gap:.2f} percentage points")
print(f"Balanced accuracy:   {balanced_acc:.2f}%")

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(
    y_val,
    combined_val_pred,
    zero_division=0
))

print("\n=== REQUIREMENTS ===")
print("Validation accuracy >= 85%:",
      "PASS" if val_acc >= 85 else "NOT YET")
print("Accuracy gap <= 5 percentage points:",
      "PASS" if gap <= 5 else "NOT YET")


=== COMBINED FEATURES LOGISTIC REGRESSION ===
Training accuracy:   83.67%
Validation accuracy: 80.24%
Accuracy gap:        3.43 percentage points
Balanced accuracy:   59.97%

=== CLASSIFICATION REPORT ===
               precision    recall  f1-score   support

      At Home       0.79      0.52      0.63        21
Dumbbell Only       0.55      0.46      0.50        13
     Full Gym       0.86      0.92      0.89       367
   Garage Gym       0.60      0.50      0.54       105

     accuracy                           0.80       506
    macro avg       0.70      0.60      0.64       506
 weighted avg       0.79      0.80      0.79       506


=== REQUIREMENTS ===
Validation accuracy >= 85%: NOT YET
Accuracy gap <= 5 percentage points: PASS


In [19]:

import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score

results = []

for k in [50, 100, 150, 200, 250, 300]:

    # Select exercise features using training data only
    selector = SelectKBest(
        score_func=chi2,
        k=k
    )

    X_train_selected = selector.fit_transform(
        X_train, y_train
    )

    X_val_selected = selector.transform(X_val)

    # Add the six workout statistics
    X_train_final = hstack([
        X_train_selected,
        csr_matrix(stats_train_scaled)
    ]).tocsr()

    X_val_final = hstack([
        X_val_selected,
        csr_matrix(stats_val_scaled)
    ]).tocsr()

    model = LogisticRegression(
        C=0.1,
        max_iter=2000,
        solver="lbfgs",
        random_state=42
    )

    model.fit(X_train_final, y_train)

    train_pred = model.predict(X_train_final)
    val_pred = model.predict(X_val_final)

    train_acc = accuracy_score(y_train, train_pred) * 100
    val_acc = accuracy_score(y_val, val_pred) * 100

    gap = abs(train_acc - val_acc)

    balanced_acc = balanced_accuracy_score(
        y_val, val_pred
    ) * 100

    results.append({
        "Number of exercise features": k,
        "Training accuracy (%)": round(train_acc, 2),
        "Validation accuracy (%)": round(val_acc, 2),
        "Gap (points)": round(gap, 2),
        "Balanced accuracy (%)": round(balanced_acc, 2),
        "Meets both requirements": (
            val_acc >= 85 and gap <= 5
        )
    })

results_df = pd.DataFrame(results)

display(results_df)


,Number of exercise features,Training accuracy (%),Validation accuracy (%),Gap (points),Balanced accuracy (%),Meets both requirements
0,50,79.81,82.41,2.60,55.50,False
1,100,80.55,80.24,0.32,54.41,False
2,150,82.53,82.21,0.32,58.91,False
3,200,83.18,80.04,3.14,61.03,False
4,250,83.37,80.43,2.94,62.06,False
5,300,83.67,80.24,3.43,59.97,False


In [20]:

import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Use the original binary exercise features plus six statistics
X_cv = hstack([
    X_train,
    csr_matrix(stats_train.to_numpy())
]).tocsr()

n_exercise_features = X_train.shape[1]
n_stat_features = stats_train.shape[1]

results = []

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for k in [50, 100, 150, 200, 250, 300]:

    # Select exercise features inside each CV fold
    preprocessing = ColumnTransformer(
        transformers=[
            (
                "exercise_selection",
                SelectKBest(chi2, k=k),
                list(range(n_exercise_features))
            ),
            (
    "workout_stats",
    StandardScaler(with_mean=False),
    list(range(
        n_exercise_features,
        n_exercise_features + n_stat_features
    ))
)
        ],
        sparse_threshold=1.0
    )

    pipeline = Pipeline([
        ("preprocessing", preprocessing),
        ("classifier", LogisticRegression(
            C=0.2,
            max_iter=2000,
            solver="lbfgs",
            random_state=42
        ))
    ])

    scores = cross_validate(
        pipeline,
        X_cv,
        y_train,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy"
        },
        return_train_score=True,
        n_jobs=1
    )

    mean_train = scores["train_accuracy"].mean() * 100
    mean_val = scores["test_accuracy"].mean() * 100

    results.append({
        "Exercise features": k,
        "CV training accuracy (%)": round(mean_train, 2),
        "CV validation accuracy (%)": round(mean_val, 2),
        "CV gap (points)": round(
            abs(mean_train - mean_val), 2
        ),
        "CV balanced accuracy (%)": round(
            scores["test_balanced_accuracy"].mean() * 100, 2
        )
    })

cv_results = pd.DataFrame(results)

display(cv_results)


,Exercise features,CV training accuracy (%),CV validation accuracy (%),CV gap (points),CV balanced accuracy (%)
0,50,79.92,76.65,3.28,50.55
1,100,81.80,78.23,3.57,56.40
2,150,83.20,78.08,5.12,56.39
3,200,84.69,78.13,6.56,58.30
4,250,85.28,78.13,7.15,57.23
5,300,85.69,77.19,8.50,55.88


In [21]:

from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate, StratifiedKFold
import pandas as pd

# Select 150 exercise features inside each CV fold
svm_preprocessing = ColumnTransformer(
    transformers=[
        (
            "exercise_selection",
            SelectKBest(chi2, k=150),
            list(range(n_exercise_features))
        ),
        (
            "workout_stats",
            StandardScaler(with_mean=False),
            list(range(
                n_exercise_features,
                n_exercise_features + n_stat_features
            ))
        )
    ],
    sparse_threshold=1.0
)

svm_pipeline = Pipeline([
    ("preprocessing", svm_preprocessing),
    ("classifier", LinearSVC(
        C=0.1,
        max_iter=10000,
        random_state=42
    ))
])

svm_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

svm_scores = cross_validate(
    svm_pipeline,
    X_cv,
    y_train,
    cv=svm_cv,
    scoring={
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy"
    },
    return_train_score=True,
    n_jobs=1
)

svm_train_acc = svm_scores["train_accuracy"].mean() * 100
svm_val_acc = svm_scores["test_accuracy"].mean() * 100
svm_gap = abs(svm_train_acc - svm_val_acc)

svm_balanced = (
    svm_scores["test_balanced_accuracy"].mean() * 100
)

print("=== LINEAR SVM CROSS-VALIDATION ===")
print(f"CV training accuracy:   {svm_train_acc:.2f}%")
print(f"CV validation accuracy: {svm_val_acc:.2f}%")
print(f"CV accuracy gap:        {svm_gap:.2f} percentage points")
print(f"CV balanced accuracy:   {svm_balanced:.2f}%")

print("\n=== REQUIREMENTS ===")
print(
    "CV validation accuracy >= 85%:",
    "PASS" if svm_val_acc >= 85 else "NOT YET"
)

print(
    "CV accuracy gap <= 5 percentage points:",
    "PASS" if svm_gap <= 5 else "NOT YET"
)


=== LINEAR SVM CROSS-VALIDATION ===
CV training accuracy:   83.96%
CV validation accuracy: 77.54%
CV accuracy gap:        6.42 percentage points
CV balanced accuracy:   56.88%

=== REQUIREMENTS ===
CV validation accuracy >= 85%: NOT YET
CV accuracy gap <= 5 percentage points: NOT YET


In [22]:

import pandas as pd

# Get unique goal labels for each program
program_goals = (
    df[["title", "goal"]]
    .dropna()
    .drop_duplicates()
)

# Keep programs with clean equipment labels
goal_equipment = program_goals.merge(
    clean_equipment,
    on="title",
    how="inner"
)

# Count programs by goal and equipment
goal_counts = pd.crosstab(
    goal_equipment["goal"],
    goal_equipment["equipment"]
)

# Show percentage within each equipment category
goal_percentages = (
    goal_counts
    .div(goal_counts.sum(axis=0), axis=1)
    * 100
).round(2)

print("=== GOAL DISTRIBUTION BY EQUIPMENT (%) ===")
display(goal_percentages)

print("\n=== NUMBER OF DISTINCT GOALS ===")
print(program_goals["goal"].nunique())


=== GOAL DISTRIBUTION BY EQUIPMENT (%) ===


equipment,At Home,Dumbbell Only,Full Gym,Garage Gym
goal,,,,
"['At-Home & Calisthenics', 'Muscle & Sculpting', 'Bodyweight Fitness']",0.00,0.00,0.03,0.00
"['Athletics', 'Bodybuilding', 'Bodyweight Fitness']",0.45,0.00,0.18,0.40
"['Athletics', 'Bodybuilding', 'Muscle & Sculpting', 'Bodyweight Fitness', 'Powerbuilding', 'Powerlifting']",0.00,0.00,0.05,0.00
"['Athletics', 'Bodybuilding', 'Muscle & Sculpting', 'Bodyweight Fitness', 'Powerbuilding']",0.00,0.00,0.10,0.10
"['Athletics', 'Bodybuilding', 'Muscle & Sculpting', 'Bodyweight Fitness']",0.45,0.00,0.23,0.10
...,...,...,...,...
"['Powerlifting', 'Powerbuilding', 'Muscle & Sculpting', 'Bodyweight Fitness', 'Olympic Weightlifting', 'Athletics', 'Bodybuilding']",0.00,0.00,0.05,0.00
"['Powerlifting', 'Powerbuilding', 'Muscle & Sculpting']",0.00,0.00,0.08,0.00
"['Powerlifting', 'Powerbuilding']",1.81,3.05,2.77,3.60



=== NUMBER OF DISTINCT GOALS ===
249


In [23]:

import ast
import pandas as pd

# Convert goal strings into actual lists
def parse_goals(value):
    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    try:
        parsed = ast.literal_eval(str(value))
        if isinstance(parsed, list):
            return parsed
    except (ValueError, SyntaxError):
        pass

    return [str(value)]

# One record per program
program_goals_clean = (
    df[["title", "goal"]]
    .drop_duplicates()
    .copy()
)

program_goals_clean["individual_goal"] = (
    program_goals_clean["goal"].apply(parse_goals)
)

# Separate individual goals into rows
program_goals_clean = (
    program_goals_clean
    .explode("individual_goal")
    .dropna(subset=["individual_goal"])
)

program_goals_clean["individual_goal"] = (
    program_goals_clean["individual_goal"]
    .astype(str)
    .str.strip()
)

program_goals_clean = (
    program_goals_clean[
        program_goals_clean["individual_goal"] != ""
    ]
    .drop_duplicates(["title", "individual_goal"])
)

# Match with clean equipment labels
goal_equipment_clean = program_goals_clean.merge(
    clean_equipment,
    on="title",
    how="inner"
)

# Count programs containing each goal
goal_counts = pd.crosstab(
    goal_equipment_clean["individual_goal"],
    goal_equipment_clean["equipment"]
)

# Percentage of programs in each equipment category
category_sizes = clean_equipment["equipment"].value_counts()

goal_percentages = (
    goal_counts.div(category_sizes, axis=1) * 100
).round(2)

print("=== INDIVIDUAL GOALS BY EQUIPMENT (%) ===")
display(goal_percentages)

print("\n=== NUMBER OF INDIVIDUAL GOALS ===")
print(program_goals_clean["individual_goal"].nunique())


=== INDIVIDUAL GOALS BY EQUIPMENT (%) ===


equipment,At Home,Dumbbell Only,Full Gym,Garage Gym
individual_goal,,,,
At-Home & Calisthenics,0.00,0.00,0.05,0.00
Athletics,31.78,37.31,37.16,35.37
Bodybuilding,85.98,77.61,84.86,82.22
Bodyweight Fitness,25.23,28.36,22.40,21.03
Muscle & Sculpting,57.94,55.22,56.01,51.05
Olympic Weightlifting,3.74,2.99,3.11,3.25
Powerbuilding,45.79,41.79,56.07,51.24
Powerlifting,29.91,32.84,37.16,34.61



=== NUMBER OF INDIVIDUAL GOALS ===
8


In [24]:

import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

# Collect the individual goals for each program
goals_by_program = (
    program_goals_clean
    .groupby("title")["individual_goal"]
    .agg(lambda x: sorted(set(x)))
    .to_dict()
)

# Align goals with our original training programs
train_goal_lists = [
    goals_by_program.get(title, [])
    for title in train_data["title"]
]

# Convert the eight goals into binary features
goal_encoder = MultiLabelBinarizer()
X_train_goals = goal_encoder.fit_transform(
    train_goal_lists
)

print("Individual goal features:", list(goal_encoder.classes_))

# Combine exercise features, workout statistics, and goals
X_cv_goals = hstack([
    X_train,
    csr_matrix(stats_train.to_numpy()),
    csr_matrix(X_train_goals)
]).tocsr()

n_exercises = X_train.shape[1]
n_stats = stats_train.shape[1]
n_goals = X_train_goals.shape[1]

preprocessing = ColumnTransformer(
    transformers=[
        (
            "exercise_selection",
            SelectKBest(chi2, k=150),
            list(range(n_exercises))
        ),
        (
            "workout_stats",
            StandardScaler(with_mean=False),
            list(range(
                n_exercises,
                n_exercises + n_stats
            ))
        ),
        (
            "goal_features",
            "passthrough",
            list(range(
                n_exercises + n_stats,
                n_exercises + n_stats + n_goals
            ))
        )
    ],
    sparse_threshold=1.0
)

goal_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("classifier", LogisticRegression(
        C=0.1,
        max_iter=2000,
        solver="lbfgs",
        random_state=42
    ))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    goal_pipeline,
    X_cv_goals,
    y_train,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy"
    },
    return_train_score=True,
    n_jobs=1
)

train_acc = scores["train_accuracy"].mean() * 100
val_acc = scores["test_accuracy"].mean() * 100
gap = abs(train_acc - val_acc)

balanced_acc = (
    scores["test_balanced_accuracy"].mean() * 100
)

print("\n=== LOGISTIC REGRESSION + GOALS ===")
print(f"CV training accuracy:   {train_acc:.2f}%")
print(f"CV validation accuracy: {val_acc:.2f}%")
print(f"CV accuracy gap:        {gap:.2f} percentage points")
print(f"CV balanced accuracy:   {balanced_acc:.2f}%")

print("\n=== REQUIREMENTS ===")
print(
    "CV validation accuracy >= 85%:",
    "PASS" if val_acc >= 85 else "NOT YET"
)
print(
    "CV accuracy gap <= 5 percentage points:",
    "PASS" if gap <= 5 else "NOT YET"
)


Individual goal features: ['At-Home & Calisthenics', 'Athletics', 'Bodybuilding', 'Bodyweight Fitness', 'Muscle & Sculpting', 'Olympic Weightlifting', 'Powerbuilding', 'Powerlifting']

=== LOGISTIC REGRESSION + GOALS ===
CV training accuracy:   82.87%
CV validation accuracy: 78.08%
CV accuracy gap:        4.79 percentage points
CV balanced accuracy:   53.22%

=== REQUIREMENTS ===
CV validation accuracy >= 85%: NOT YET
CV accuracy gap <= 5 percentage points: PASS


In [25]:
import pandas as pd

print("=== DATASET COLUMNS ===")
print(df.columns.tolist())

print("\n=== SAMPLE VALUES BY COLUMN ===")

for column in df.columns:
    print(f"\n{column}")

    # Convert columns containing lists to tuples so they can be hashed for nunique and drop_duplicates
    has_lists = df[column].apply(lambda x: isinstance(x, list)).any()
    if has_lists:
        series_to_analyze = df[column].apply(lambda x: tuple(x) if isinstance(x, list) else x)
    else:
        series_to_analyze = df[column]

    print("Unique values:", series_to_analyze.nunique(dropna=True))
    print(
        series_to_analyze
        .dropna()
        .astype(str)
        .drop_duplicates()
        .head(5)
        .tolist()
    )

=== DATASET COLUMNS ===
['title', 'description', 'level', 'goal', 'equipment', 'program_length', 'time_per_workout', 'week', 'day', 'number_of_exercises', 'exercise_name', 'sets', 'reps', 'intensity', 'created', 'last_edit', 'parsed_level']

=== SAMPLE VALUES BY COLUMN ===

title
Unique values: 2537
['Lyle McDonald Routine (Strength/Hypertrophy Vers)', 'Jeff Nippard’s Fundamentals, Upper&Lower Split, 4x', 'nSuns_Rebuild by KSI', 'Rampage', 'Fitness King Powerbuild 101']

description
Unique values: 2472
['IMPORTANT: THIS IS NOT THE ORIGINAL LYLE MCDONALD\'S SCHEDULE. THIS IS THE SCHEDULE THAT DESIGNED BY A USER WITH BASING LYLE MCDONALD\'S SCHEDULE. BUT ALSO THIS SCHEDULE IS VERY SIMILAR TO LYLE MCDONALD\'S SCHEDULE AND THE DESCRIPTION WHICH IN THE BELOW IS WROTE BY LYLE MCDONALD AND LIFT VAULT. IF YOU WANT TO LOOK ORIGINAL PROGRAM YOU CAN VISIT THE LIFT VAULT WEBSITE.\nTHANK YOU!\n\nLyle McDonald’s generic bulking routine allows for a lot of flexibility within the program. If you are n

In [26]:

import pandas as pd

# Get one duration value per program
duration_data = (
    df[["title", "time_per_workout"]]
    .drop_duplicates()
    .copy()
)

duration_data["time_per_workout"] = pd.to_numeric(
    duration_data["time_per_workout"],
    errors="coerce"
)

# Count distinct duration values per program
duration_counts = (
    duration_data.groupby("title")["time_per_workout"]
    .nunique(dropna=True)
)

print("=== DURATION LABEL CONSISTENCY ===")
print("Programs with one duration:",
      (duration_counts == 1).sum())
print("Programs with multiple durations:",
      (duration_counts > 1).sum())
print("Programs with missing duration:",
      (duration_counts == 0).sum())

print("\n=== WORKOUT DURATION DISTRIBUTION ===")
print(
    duration_data["time_per_workout"]
    .value_counts(dropna=False)
    .sort_index()
)


=== DURATION LABEL CONSISTENCY ===
Programs with one duration: 2514
Programs with multiple durations: 23
Programs with missing duration: 0

=== WORKOUT DURATION DISTRIBUTION ===
time_per_workout
10.0      28
20.0      30
30.0      81
40.0     169
50.0     205
60.0     822
70.0     334
72.0       2
80.0     223
90.0     422
100.0     70
110.0     12
120.0    133
130.0      5
140.0      5
150.0      7
160.0      2
170.0      1
180.0     11
Name: count, dtype: int64


In [27]:

import pandas as pd
import numpy as np

# Keep programs with exactly one known duration
duration_counts = (
    duration_data.groupby("title")["time_per_workout"]
    .nunique(dropna=True)
)

valid_duration_titles = duration_counts[
    duration_counts == 1
].index

clean_duration = (
    duration_data[
        duration_data["title"].isin(valid_duration_titles)
        & duration_data["time_per_workout"].notna()
    ]
    .drop_duplicates(subset=["title"])
    .copy()
)

# Define meaningful duration categories
clean_duration["duration_category"] = pd.cut(
    clean_duration["time_per_workout"],
    bins=[-np.inf, 60, 90, np.inf],
    labels=["Short (<=60)", "Medium (61-90)", "Long (>90)"],
    right=True
)

print("=== DURATION CATEGORY DISTRIBUTION ===")
print(
    clean_duration["duration_category"]
    .value_counts()
)

print("\n=== DURATION CATEGORY PERCENTAGES ===")
print(
    clean_duration["duration_category"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\n=== TOTAL PROGRAMS ===")
print(len(clean_duration))

print("\n=== MAJORITY-CLASS BASELINE ===")
print(
    f"{clean_duration['duration_category'].value_counts(normalize=True).max() * 100:.2f}%"
)


=== DURATION CATEGORY DISTRIBUTION ===
duration_category
Short (<=60)      1313
Medium (61-90)     960
Long (>90)         241
Name: count, dtype: int64

=== DURATION CATEGORY PERCENTAGES ===
duration_category
Short (<=60)      52.23
Medium (61-90)    38.19
Long (>90)         9.59
Name: proportion, dtype: float64

=== TOTAL PROGRAMS ===
2514

=== MAJORITY-CLASS BASELINE ===
52.23%


In [28]:

import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

# Match clean duration labels to program exercise lists
duration_model_data = clean_duration.merge(
    program_exercises,
    on="title",
    how="inner",
    validate="one_to_one"
)

# Encode exercise presence
duration_encoder = MultiLabelBinarizer(sparse_output=True)

X_duration_exercises = duration_encoder.fit_transform(
    duration_model_data["exercises"]
)

# Add workout structure statistics
# Do NOT include time_per_workout or duration-derived features
duration_stat_columns = [
    "total_exercise_entries",
    "unique_exercises",
    "average_sets",
    "average_program_length",
    "total_workout_days",
    "exercises_per_day"
]

duration_stats = workout_stats.reindex(
    duration_model_data["title"]
)[duration_stat_columns]

duration_stats = duration_stats.replace(
    [np.inf, -np.inf], np.nan
)

# Missing-value handling for this initial experiment
duration_stats = duration_stats.fillna(
    duration_stats.median().fillna(0)
)

X_duration = hstack([
    X_duration_exercises,
    csr_matrix(duration_stats.to_numpy())
]).tocsr()

y_duration = (
    duration_model_data["duration_category"]
    .astype(str)
    .to_numpy()
)

n_exercises = X_duration_exercises.shape[1]
n_stats = duration_stats.shape[1]

# Select exercise features within each CV fold
duration_preprocessing = ColumnTransformer(
    transformers=[
        (
            "exercise_selection",
            SelectKBest(
                score_func=chi2,
                k=min(150, n_exercises)
            ),
            list(range(n_exercises))
        ),
        (
            "workout_stats",
            StandardScaler(with_mean=False),
            list(range(
                n_exercises,
                n_exercises + n_stats
            ))
        )
    ],
    sparse_threshold=1.0
)

duration_pipeline = Pipeline([
    ("preprocessing", duration_preprocessing),
    ("classifier", LogisticRegression(
        C=0.1,
        max_iter=2000,
        solver="lbfgs",
        random_state=42
    ))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    duration_pipeline,
    X_duration,
    y_duration,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy"
    },
    return_train_score=True,
    n_jobs=1
)

train_acc = scores["train_accuracy"].mean() * 100
val_acc = scores["test_accuracy"].mean() * 100
gap = abs(train_acc - val_acc)

balanced_acc = (
    scores["test_balanced_accuracy"].mean() * 100
)

print("=== WORKOUT DURATION CLASSIFICATION ===")
print(f"Programs evaluated:    {len(y_duration)}")
print(f"CV training accuracy:   {train_acc:.2f}%")
print(f"CV validation accuracy: {val_acc:.2f}%")
print(f"CV accuracy gap:        {gap:.2f} percentage points")
print(f"CV balanced accuracy:   {balanced_acc:.2f}%")

print("\n=== REQUIREMENTS ===")
print(
    "CV validation accuracy >= 85%:",
    "PASS" if val_acc >= 85 else "NOT YET"
)
print(
    "CV accuracy gap <= 5 percentage points:",
    "PASS" if gap <= 5 else "NOT YET"
)


=== WORKOUT DURATION CLASSIFICATION ===
Programs evaluated:    2514
CV training accuracy:   65.28%
CV validation accuracy: 57.88%
CV accuracy gap:        7.41 percentage points
CV balanced accuracy:   42.21%

=== REQUIREMENTS ===
CV validation accuracy >= 85%: NOT YET
CV accuracy gap <= 5 percentage points: NOT YET


In [29]:

# List all distinct exercises in the recreated dataset

exercise_names = sorted(
    df["exercise_name"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print("=== TOTAL DISTINCT EXERCISES ===")
print(len(exercise_names))

print("\n=== EXERCISE NAMES ===")
for index, exercise in enumerate(exercise_names, start=1):
    print(f"{index:3}. {exercise}")


=== TOTAL DISTINCT EXERCISES ===
300

=== EXERCISE NAMES ===
  1. 21s (EZ Bar)
  2. AD Press
  3. Ab Wheel
  4. Abs Crunch (Bodyweight)
  5. Abs Crunch (Machine)
  6. Abs Crunch (Weighted)
  7. Alternating Dumbbell Curl
  8. Arnold Press
  9. Back Extension
 10. Back Extension (Weighted)
 11. Band Pull Apart
 12. Barbell Row
 13. Bayesian Curl
 14. Belt Squat
 15. Bench Press (Barbell)
 16. Bench Press (Close Grip)
 17. Bench Press (Dumbbell)
 18. Bench Press (Paused)
 19. Bench Press (Smith Machine)
 20. Bench Press (Wide Grip)
 21. Bent Over Row (Barbell)
 22. Bent Over Row (Dumbbell)
 23. Bicep Curl (Barbell)
 24. Bicep Curl (Cable)
 25. Bicep Curl (Dumbbell)
 26. Bicep Curl (EZ Bar)
 27. Bicep Curl (Machine)
 28. Bicycle Crunch
 29. Box Jump
 30. Box Squat (Barbell)
 31. Bulgarian Split Squat (Barbell)
 32. Bulgarian Split Squat (Bodyweight)
 33. Bulgarian Split Squat (Dumbbell)
 34. Burpee
 35. Cable Crossover
 36. Cable Crunch
 37. Cable Low Row
 38. Calf Raise (Bodyweight)
 39. 

In [30]:

import pandas as pd
import re

# Detect explicit equipment words in exercise names
equipment_patterns = {
    "Barbell": r"\bbarbell\b",
    "Dumbbell": r"\bdumbbell\b",
    "Cable": r"\bcable\b",
    "Machine": r"\bmachine\b",
    "Smith Machine": r"\bsmith machine\b",
    "Bodyweight": r"\bbodyweight\b",
    "EZ Bar": r"\bez bar\b",
    "Band": r"\bband\b",
    "Kettlebell": r"\bkettlebell\b"
}

exercise_equipment_analysis = pd.DataFrame({
    "exercise_name": exercise_names
})

for equipment, pattern in equipment_patterns.items():
    exercise_equipment_analysis[equipment] = (
        exercise_equipment_analysis["exercise_name"]
        .str.contains(pattern, case=False, regex=True)
    )

# Count how many different equipment keywords appear
equipment_columns = list(equipment_patterns.keys())

exercise_equipment_analysis["equipment_keyword_count"] = (
    exercise_equipment_analysis[equipment_columns].sum(axis=1)
)

print("=== EQUIPMENT KEYWORD COVERAGE ===")

print(
    "Exercises with at least one equipment keyword:",
    (exercise_equipment_analysis["equipment_keyword_count"] > 0).sum()
)

print(
    "Exercises without equipment keywords:",
    (exercise_equipment_analysis["equipment_keyword_count"] == 0).sum()
)

print("\n=== KEYWORD FREQUENCY ===")

print(
    exercise_equipment_analysis[equipment_columns]
    .sum()
    .sort_values(ascending=False)
)

print("\n=== EXERCISES WITHOUT EXPLICIT EQUIPMENT KEYWORDS ===")

display(
    exercise_equipment_analysis.loc[
        exercise_equipment_analysis["equipment_keyword_count"] == 0,
        ["exercise_name"]
    ].reset_index(drop=True)
)


=== EQUIPMENT KEYWORD COVERAGE ===
Exercises with at least one equipment keyword: 158
Exercises without equipment keywords: 142

=== KEYWORD FREQUENCY ===
Dumbbell         50
Barbell          39
Cable            25
Machine          25
Bodyweight       12
Smith Machine     6
EZ Bar            4
Band              2
Kettlebell        1
dtype: int64

=== EXERCISES WITHOUT EXPLICIT EQUIPMENT KEYWORDS ===


,exercise_name
0,AD Press
1,Ab Wheel
2,Abs Crunch (Weighted)
3,Arnold Press
4,Back Extension
...,...
137,Wide Grip Lat Pulldown
138,Wide Grip Pull-Up
139,Wood Chop
140,Wrist Curls


In [31]:

import ast
import pandas as pd

def parse_levels(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(str(value))
        if isinstance(parsed, list):
            return parsed
    except (ValueError, SyntaxError):
        pass
    return [str(value)]

program_levels = (
    df[["title", "level"]]
    .drop_duplicates()
    .copy()
)

program_levels["parsed_levels"] = (
    program_levels["level"].apply(parse_levels)
)

# Combine all distinct difficulty labels for each program
levels_by_program = (
    program_levels.groupby("title")["parsed_levels"]
    .agg(
        lambda rows: sorted({
            str(level).strip()
            for row in rows
            for level in row
            if str(level).strip()
        })
    )
)

level_counts = levels_by_program.apply(len)

print("=== DIFFICULTY LABEL COUNTS ===")
print(level_counts.value_counts().sort_index())

single_level = levels_by_program[
    level_counts == 1
].apply(lambda levels: levels[0])

print("\n=== SINGLE-LABEL PROGRAM DISTRIBUTION ===")
print(single_level.value_counts())

print("\n=== TOTAL SINGLE-LABEL PROGRAMS ===")
print(len(single_level))

print("\n=== MAJORITY-CLASS BASELINE ===")
if len(single_level) > 0:
    print(
        f"{single_level.value_counts(normalize=True).max() * 100:.2f}%"
    )


=== DIFFICULTY LABEL COUNTS ===
parsed_levels
0      2
1    461
2    806
3    707
4    561
Name: count, dtype: int64

=== SINGLE-LABEL PROGRAM DISTRIBUTION ===
parsed_levels
Intermediate    280
Novice           85
Advanced         50
Beginner         46
Name: count, dtype: int64

=== TOTAL SINGLE-LABEL PROGRAMS ===
461

=== MAJORITY-CLASS BASELINE ===
60.74%


In [32]:

import pandas as pd

dataset_url = (
    "https://raw.githubusercontent.com/"
    "yuhonas/free-exercise-db/main/dist/exercises.json"
)

exercise_df = pd.read_json(dataset_url)

print("=== EXTERNAL EXERCISE DATASET ===")
print("Total exercises:", len(exercise_df))

print("\n=== AVAILABLE COLUMNS ===")
print(exercise_df.columns.tolist())

print("\n=== SAMPLE EXERCISES ===")
display(
    exercise_df[
        ["name", "primaryMuscles", "secondaryMuscles",
         "equipment", "category"]
    ].head(10)
)

print("\n=== MISSING PRIMARY MUSCLE LABELS ===")
print(exercise_df["primaryMuscles"].isna().sum())


=== EXTERNAL EXERCISE DATASET ===
Total exercises: 876

=== AVAILABLE COLUMNS ===
['name', 'force', 'level', 'mechanic', 'equipment', 'primaryMuscles', 'secondaryMuscles', 'instructions', 'category', 'images', 'id']

=== SAMPLE EXERCISES ===


,name,primaryMuscles,secondaryMuscles,equipment,category
0,3/4 Sit-Up,[abdominals],[],body only,strength
1,90/90 Hamstring,[hamstrings],[calves],body only,stretching
2,Ab Crunch Machine,[abdominals],[],machine,strength
3,Ab Roller,[abdominals],[shoulders],other,strength
4,Adductor,[adductors],[],foam roll,stretching
5,Adductor/Groin,[adductors],[],None,stretching
6,Advanced Kettlebell Windmill,[abdominals],"[glutes, hamstrings, shoulders]",kettlebells,strength
7,Air Bike,[abdominals],[],body only,strength
8,All Fours Quad Stretch,[quadriceps],[quadriceps],body only,stretching
9,Alternate Hammer Curl,[biceps],[forearms],dumbbell,strength



=== MISSING PRIMARY MUSCLE LABELS ===
0


In [33]:

import pandas as pd

# Count primary muscle labels per exercise
exercise_df["primary_muscle_count"] = (
    exercise_df["primaryMuscles"].apply(
        lambda muscles: len(muscles)
        if isinstance(muscles, list) else 0
    )
)

print("=== PRIMARY MUSCLE LABEL COUNTS ===")
print(
    exercise_df["primary_muscle_count"]
    .value_counts()
    .sort_index()
)

# Keep exercises with exactly one primary muscle
single_muscle_df = exercise_df[
    exercise_df["primary_muscle_count"] == 1
].copy()

single_muscle_df["primary_muscle"] = (
    single_muscle_df["primaryMuscles"]
    .apply(lambda muscles: muscles[0])
)

print("\n=== PRIMARY MUSCLE CLASS DISTRIBUTION ===")
print(
    single_muscle_df["primary_muscle"]
    .value_counts()
)

print("\n=== TOTAL SINGLE-LABEL EXERCISES ===")
print(len(single_muscle_df))

print("\n=== NUMBER OF MUSCLE CLASSES ===")
print(single_muscle_df["primary_muscle"].nunique())

print("\n=== MAJORITY-CLASS BASELINE ===")
print(
    f"{single_muscle_df['primary_muscle'].value_counts(normalize=True).max() * 100:.2f}%"
)


=== PRIMARY MUSCLE LABEL COUNTS ===
primary_muscle_count
1    875
2      1
Name: count, dtype: int64

=== PRIMARY MUSCLE CLASS DISTRIBUTION ===
primary_muscle
quadriceps     148
shoulders      128
abdominals      93
chest           84
hamstrings      79
triceps         72
biceps          53
lats            38
middle back     34
calves          28
lower back      27
forearms        25
glutes          22
traps           15
adductors       13
neck             8
abductors        8
Name: count, dtype: int64

=== TOTAL SINGLE-LABEL EXERCISES ===
875

=== NUMBER OF MUSCLE CLASSES ===
17

=== MAJORITY-CLASS BASELINE ===
16.91%


In [34]:

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# Build input text from exercise name and instructions
muscle_data = single_muscle_df.copy()

muscle_data["instruction_text"] = (
    muscle_data["instructions"]
    .apply(
        lambda steps: " ".join(steps)
        if isinstance(steps, list) else ""
    )
)

muscle_data["model_text"] = (
    muscle_data["name"].fillna("")
    + " "
    + muscle_data["instruction_text"]
)

X = muscle_data["model_text"]
y = muscle_data["primary_muscle"]

# Stratified train-validation split
X_train_muscle, X_val_muscle, y_train_muscle, y_val_muscle = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

# TF-IDF + Logistic Regression
muscle_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2
    )),
    ("classifier", LogisticRegression(
        C=1.0,
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ))
])

muscle_pipeline.fit(
    X_train_muscle,
    y_train_muscle
)

train_predictions = muscle_pipeline.predict(
    X_train_muscle
)

val_predictions = muscle_pipeline.predict(
    X_val_muscle
)

train_accuracy = accuracy_score(
    y_train_muscle,
    train_predictions
) * 100

val_accuracy = accuracy_score(
    y_val_muscle,
    val_predictions
) * 100

accuracy_gap = abs(train_accuracy - val_accuracy)

balanced_accuracy = balanced_accuracy_score(
    y_val_muscle,
    val_predictions
) * 100

print("=== PRIMARY MUSCLE CLASSIFICATION ===")
print(f"Training exercises:     {len(X_train_muscle)}")
print(f"Validation exercises:   {len(X_val_muscle)}")
print(f"Training accuracy:      {train_accuracy:.2f}%")
print(f"Validation accuracy:    {val_accuracy:.2f}%")
print(f"Accuracy gap:           {accuracy_gap:.2f} percentage points")
print(f"Balanced accuracy:      {balanced_accuracy:.2f}%")

print("\n=== PROFESSOR'S REQUIREMENTS ===")
print(
    "Validation accuracy >= 85%:",
    "PASS" if val_accuracy >= 85 else "NOT YET"
)
print(
    "Accuracy gap <= 5 percentage points:",
    "PASS" if accuracy_gap <= 5 else "NOT YET"
)


=== PRIMARY MUSCLE CLASSIFICATION ===
Training exercises:     700
Validation exercises:   175
Training accuracy:      92.57%
Validation accuracy:    67.43%
Accuracy gap:           25.14 percentage points
Balanced accuracy:      64.16%

=== PROFESSOR'S REQUIREMENTS ===
Validation accuracy >= 85%: NOT YET
Accuracy gap <= 5 percentage points: NOT YET


In [35]:

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for c_value in [0.01, 0.03, 0.1, 0.3, 1.0]:

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            min_df=2
        )),
        ("classifier", LogisticRegression(
            C=c_value,
            max_iter=3000,
            class_weight="balanced",
            random_state=42
        ))
    ])

    scores = cross_validate(
        pipeline,
        X_train_muscle,
        y_train_muscle,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy"
        },
        return_train_score=True,
        n_jobs=1
    )

    train_acc = scores["train_accuracy"].mean() * 100
    val_acc = scores["test_accuracy"].mean() * 100

    results.append({
        "C": c_value,
        "CV Training Accuracy": round(train_acc, 2),
        "CV Validation Accuracy": round(val_acc, 2),
        "Gap (points)": round(abs(train_acc - val_acc), 2),
        "Balanced Accuracy": round(
            scores["test_balanced_accuracy"].mean() * 100,
            2
        )
    })

results_df = pd.DataFrame(results)

print("=== REGULARIZATION COMPARISON ===")
display(results_df)


=== REGULARIZATION COMPARISON ===


,C,CV Training Accuracy,CV Validation Accuracy,Gap (points),Balanced Accuracy
0,0.01,77.11,49.71,27.39,54.68
1,0.03,77.64,50.29,27.36,54.79
2,0.10,79.71,52.43,27.29,55.71
3,0.30,84.71,56.57,28.14,56.26
4,1.00,92.14,61.14,31.00,53.65


In [36]:

import pandas as pd
from sklearn.metrics import confusion_matrix

# Use predictions from our original Step 30 model
# on the 175 held-out validation exercises
labels = sorted(y_val_muscle.unique())

cm = confusion_matrix(
    y_val_muscle,
    val_predictions,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

# Show the most frequent incorrect predictions
errors = []

for actual in labels:
    for predicted in labels:
        if actual != predicted:
            count = cm_df.loc[actual, predicted]
            if count > 0:
                errors.append({
                    "Actual Muscle": actual,
                    "Predicted Muscle": predicted,
                    "Count": count
                })

errors_df = (
    pd.DataFrame(errors)
    .sort_values("Count", ascending=False)
    .reset_index(drop=True)
)

print("=== TOP 20 MUSCLE CLASSIFICATION ERRORS ===")
display(errors_df.head(20))

print("\n=== TOTAL VALIDATION ERRORS ===")
print((y_val_muscle.to_numpy() != val_predictions).sum())


=== TOP 20 MUSCLE CLASSIFICATION ERRORS ===


,Actual Muscle,Predicted Muscle,Count
0,hamstrings,quadriceps,4
1,shoulders,hamstrings,3
2,quadriceps,hamstrings,3
3,chest,shoulders,2
4,forearms,triceps,2
5,hamstrings,adductors,2
6,lower back,abdominals,2
7,hamstrings,glutes,2
8,shoulders,traps,2
9,abdominals,calves,1



=== TOTAL VALIDATION ERRORS ===
57


In [37]:

import pandas as pd

# Define broader muscle regions before model training
muscle_region_map = {
    "chest": "Upper Body",
    "shoulders": "Upper Body",
    "triceps": "Upper Body",
    "biceps": "Upper Body",
    "lats": "Upper Body",
    "middle back": "Upper Body",
    "traps": "Upper Body",
    "forearms": "Upper Body",

    "quadriceps": "Lower Body",
    "hamstrings": "Lower Body",
    "glutes": "Lower Body",
    "calves": "Lower Body",
    "adductors": "Lower Body",
    "abductors": "Lower Body",

    "abdominals": "Core",
    "lower back": "Core",

    "neck": "Neck"
}

region_data = muscle_data.copy()

region_data["muscle_region"] = (
    region_data["primary_muscle"]
    .map(muscle_region_map)
)

print("=== MUSCLE REGION DISTRIBUTION ===")
print(region_data["muscle_region"].value_counts())

print("\n=== MUSCLE REGION PERCENTAGES ===")
print(
    region_data["muscle_region"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\n=== UNMAPPED MUSCLES ===")
print(
    region_data.loc[
        region_data["muscle_region"].isna(),
        "primary_muscle"
    ].unique()
)

print("\n=== MAJORITY-CLASS BASELINE ===")
print(
    f"{region_data['muscle_region'].value_counts(normalize=True).max() * 100:.2f}%"
)


=== MUSCLE REGION DISTRIBUTION ===
muscle_region
Upper Body    449
Lower Body    298
Core          120
Neck            8
Name: count, dtype: int64

=== MUSCLE REGION PERCENTAGES ===
muscle_region
Upper Body    51.31
Lower Body    34.06
Core          13.71
Neck           0.91
Name: proportion, dtype: float64

=== UNMAPPED MUSCLES ===
[]

=== MAJORITY-CLASS BASELINE ===
51.31%


In [38]:

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

# Use only the original 700 training exercises
# Keep the same rows as our original muscle classifier
X_train_region = muscle_data.loc[
    X_train_muscle.index, "model_text"
]

y_train_region = region_data.loc[
    X_train_muscle.index, "muscle_region"
]

print("=== TRAINING REGION DISTRIBUTION ===")
print(y_train_region.value_counts())

region_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 1),
        min_df=2
    )),
    ("classifier", LogisticRegression(
        C=0.5,
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    region_pipeline,
    X_train_region,
    y_train_region,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy"
    },
    return_train_score=True,
    n_jobs=1
)

train_acc = scores["train_accuracy"].mean() * 100
val_acc = scores["test_accuracy"].mean() * 100
gap = abs(train_acc - val_acc)

balanced_acc = (
    scores["test_balanced_accuracy"].mean() * 100
)

print("\n=== FOUR-REGION CLASSIFICATION ===")
print(f"CV training accuracy:   {train_acc:.2f}%")
print(f"CV validation accuracy: {val_acc:.2f}%")
print(f"CV accuracy gap:        {gap:.2f} percentage points")
print(f"CV balanced accuracy:   {balanced_acc:.2f}%")

print("\n=== PROFESSOR'S REQUIREMENTS ===")
print(
    "CV validation accuracy >= 85%:",
    "PASS" if val_acc >= 85 else "NOT YET"
)
print(
    "CV accuracy gap <= 5 percentage points:",
    "PASS" if gap <= 5 else "NOT YET"
)


=== TRAINING REGION DISTRIBUTION ===
muscle_region
Upper Body    359
Lower Body    238
Core           97
Neck            6
Name: count, dtype: int64

=== FOUR-REGION CLASSIFICATION ===
CV training accuracy:   92.11%
CV validation accuracy: 84.14%
CV accuracy gap:        7.96 percentage points
CV balanced accuracy:   77.44%

=== PROFESSOR'S REQUIREMENTS ===
CV validation accuracy >= 85%: NOT YET
CV accuracy gap <= 5 percentage points: NOT YET


In [39]:

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for c_value in [0.01, 0.03, 0.1, 0.3, 0.5, 1.0]:

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 1),
            min_df=2
        )),
        ("classifier", LogisticRegression(
            C=c_value,
            max_iter=3000,
            class_weight="balanced",
            random_state=42
        ))
    ])

    scores = cross_validate(
        pipeline,
        X_train_region,
        y_train_region,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy"
        },
        return_train_score=True,
        n_jobs=1
    )

    train_acc = scores["train_accuracy"].mean() * 100
    val_acc = scores["test_accuracy"].mean() * 100
    gap = abs(train_acc - val_acc)

    results.append({
        "C": c_value,
        "CV Training Accuracy": round(train_acc, 2),
        "CV Validation Accuracy": round(val_acc, 2),
        "Gap (points)": round(gap, 2),
        "Balanced Accuracy": round(
            scores["test_balanced_accuracy"].mean() * 100,
            2
        ),
        "Meets Both": (
            "YES" if val_acc >= 85 and gap <= 5
            else "NO"
        )
    })

results_df = pd.DataFrame(results)

print("=== FOUR-REGION REGULARIZATION COMPARISON ===")
display(results_df)


=== FOUR-REGION REGULARIZATION COMPARISON ===


,C,CV Training Accuracy,CV Validation Accuracy,Gap (points),Balanced Accuracy,Meets Both
0,0.01,85.86,80.57,5.29,74.53,NO
1,0.03,86.36,80.86,5.50,74.89,NO
2,0.10,87.96,81.86,6.11,75.90,NO
3,0.30,90.75,83.29,7.46,77.22,NO
4,0.50,92.11,84.14,7.96,77.44,NO
5,1.00,94.64,84.86,9.79,77.86,NO


In [40]:

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

feature_settings = [
    {
        "name": "Single words",
        "ngram_range": (1, 1),
        "min_df": 2
    },
    {
        "name": "Single + two-word phrases",
        "ngram_range": (1, 2),
        "min_df": 2
    },
    {
        "name": "Single words, min_df=5",
        "ngram_range": (1, 1),
        "min_df": 5
    }
]

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for setting in feature_settings:

    pipeline = Pipeline([
       ("tfidf", TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=setting["ngram_range"],
    min_df=setting["min_df"],
    max_features=300
)),
        ("classifier", LogisticRegression(
            C=0.5,
            max_iter=3000,
            class_weight="balanced",
            random_state=42
        ))
    ])

    scores = cross_validate(
        pipeline,
        X_train_region,
        y_train_region,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy"
        },
        return_train_score=True,
        n_jobs=1
    )

    train_acc = scores["train_accuracy"].mean() * 100
    val_acc = scores["test_accuracy"].mean() * 100
    gap = abs(train_acc - val_acc)

    results.append({
        "Features": setting["name"],
        "CV Training Accuracy": round(train_acc, 2),
        "CV Validation Accuracy": round(val_acc, 2),
        "Gap (points)": round(gap, 2),
        "Balanced Accuracy": round(
            scores["test_balanced_accuracy"].mean() * 100,
            2
        ),
        "Meets Both": (
            "YES" if val_acc >= 85 and gap <= 5
            else "NO"
        )
    })

print("=== TEXT FEATURE COMPARISON ===")
display(pd.DataFrame(results))


=== TEXT FEATURE COMPARISON ===


,Features,CV Training Accuracy,CV Validation Accuracy,Gap (points),Balanced Accuracy,Meets Both
0,Single words,89.43,84.00,5.43,77.92,NO
1,Single + two-word phrases,88.57,82.57,6.00,75.86,NO
2,"Single words, min_df=5",89.43,84.14,5.29,77.99,NO


In [41]:

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for max_features in [100, 200, 300, 500, None]:

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 1),
            min_df=5,
            max_features=max_features
        )),
        ("classifier", LogisticRegression(
            C=0.5,
            max_iter=3000,
            class_weight="balanced",
            random_state=42
        ))
    ])

    scores = cross_validate(
        pipeline,
        X_train_region,
        y_train_region,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy"
        },
        return_train_score=True,
        n_jobs=1
    )

    train_acc = scores["train_accuracy"].mean() * 100
    val_acc = scores["test_accuracy"].mean() * 100
    gap = abs(train_acc - val_acc)

    results.append({
        "Max Features": (
            str(max_features)
            if max_features is not None
            else "No limit"
        ),
        "CV Training Accuracy": round(train_acc, 2),
        "CV Validation Accuracy": round(val_acc, 2),
        "Gap (points)": round(gap, 2),
        "Balanced Accuracy": round(
            scores["test_balanced_accuracy"].mean() * 100,
            2
        ),
        "Meets Both": (
            "YES" if val_acc >= 85 and gap <= 5
            else "NO"
        )
    })

print("=== VOCABULARY SIZE COMPARISON ===")
display(pd.DataFrame(results))


=== VOCABULARY SIZE COMPARISON ===


,Max Features,CV Training Accuracy,CV Validation Accuracy,Gap (points),Balanced Accuracy,Meets Both
0,100,85.07,79.29,5.79,77.82,NO
1,200,87.93,82.57,5.36,76.27,NO
2,300,89.43,84.14,5.29,77.99,NO
3,500,90.71,84.00,6.71,77.53,NO
4,No limit,91.32,84.43,6.89,77.81,NO


In [42]:

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

models = {
    "Logistic Regression": LogisticRegression(
        C=0.5,
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ),
    "Naive Bayes (alpha=0.1)": MultinomialNB(alpha=0.1),
    "Naive Bayes (alpha=1.0)": MultinomialNB(alpha=1.0),
    "Naive Bayes (alpha=5.0)": MultinomialNB(alpha=5.0)
}

results = []

for model_name, classifier in models.items():

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 1),
            min_df=5,
            max_features=300
        )),
        ("classifier", classifier)
    ])

    scores = cross_validate(
        pipeline,
        X_train_region,
        y_train_region,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy"
        },
        return_train_score=True,
        n_jobs=1
    )

    train_acc = scores["train_accuracy"].mean() * 100
    val_acc = scores["test_accuracy"].mean() * 100
    gap = abs(train_acc - val_acc)

    results.append({
        "Model": model_name,
        "CV Training Accuracy": round(train_acc, 2),
        "CV Validation Accuracy": round(val_acc, 2),
        "Gap (points)": round(gap, 2),
        "Balanced Accuracy": round(
            scores["test_balanced_accuracy"].mean() * 100,
            2
        ),
        "Meets Both": (
            "YES" if val_acc >= 85 and gap <= 5
            else "NO"
        )
    })

print("=== CLASSIFIER COMPARISON ===")
display(pd.DataFrame(results))


=== CLASSIFIER COMPARISON ===


,Model,CV Training Accuracy,CV Validation Accuracy,Gap (points),Balanced Accuracy,Meets Both
0,Logistic Regression,89.43,84.14,5.29,77.99,NO
1,Naive Bayes (alpha=0.1),85.29,78.86,6.43,55.72,NO
2,Naive Bayes (alpha=1.0),81.25,77.00,4.25,45.79,NO
3,Naive Bayes (alpha=5.0),77.64,76.57,1.07,44.11,NO


In [43]:

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report
)

# Labels for the original 175 held-out exercises
X_val_region = muscle_data.loc[
    X_val_muscle.index, "model_text"
]

y_val_region = region_data.loc[
    X_val_muscle.index, "muscle_region"
]

# Select the balanced model from our CV comparisons
final_region_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 1),
        min_df=5,
        max_features=300
    )),
    ("classifier", LogisticRegression(
        C=0.5,
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ))
])

# Fit using training data only
final_region_model.fit(
    X_train_region,
    y_train_region
)

train_predictions = final_region_model.predict(
    X_train_region
)

val_predictions_region = final_region_model.predict(
    X_val_region
)

train_accuracy = accuracy_score(
    y_train_region,
    train_predictions
) * 100

val_accuracy = accuracy_score(
    y_val_region,
    val_predictions_region
) * 100

gap = abs(train_accuracy - val_accuracy)

balanced_accuracy = balanced_accuracy_score(
    y_val_region,
    val_predictions_region
) * 100

print("=== HELD-OUT EVALUATION ===")
print(f"Training accuracy:   {train_accuracy:.2f}%")
print(f"Validation accuracy: {val_accuracy:.2f}%")
print(f"Accuracy gap:        {gap:.2f} percentage points")
print(f"Balanced accuracy:   {balanced_accuracy:.2f}%")

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(
    y_val_region,
    val_predictions_region,
    zero_division=0
))

print("\n=== PROFESSOR'S REQUIREMENTS ===")
print(
    "Validation accuracy >= 85%:",
    "PASS" if val_accuracy >= 85 else "NOT YET"
)
print(
    "Accuracy gap <= 5 points:",
    "PASS" if gap <= 5 else "NOT YET"
)


=== HELD-OUT EVALUATION ===
Training accuracy:   88.57%
Validation accuracy: 90.86%
Accuracy gap:        2.29 percentage points
Balanced accuracy:   90.95%

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

        Core       0.64      0.78      0.71        23
  Lower Body       0.93      0.93      0.93        60
        Neck       0.67      1.00      0.80         2
  Upper Body       0.99      0.92      0.95        90

    accuracy                           0.91       175
   macro avg       0.81      0.91      0.85       175
weighted avg       0.92      0.91      0.91       175


=== PROFESSOR'S REQUIREMENTS ===
Validation accuracy >= 85%: PASS
Accuracy gap <= 5 points: PASS


In [44]:

import joblib
import json
from google.colab import files

# Save the trained TF-IDF + Logistic Regression pipeline
model_filename = "fitroutine_muscle_region_classifier.joblib"

joblib.dump(final_region_model, model_filename)

# Save the model's class labels and evaluation details
model_info = {
    "model": "TF-IDF + Logistic Regression",
    "task": "Exercise muscle-region classification",
    "classes": final_region_model.classes_.tolist(),
    "training_exercises": len(X_train_region),
    "validation_exercises": len(X_val_region),
    "training_accuracy": round(train_accuracy, 2),
    "validation_accuracy": round(val_accuracy, 2),
    "accuracy_gap_points": round(gap, 2),
    "balanced_accuracy": round(balanced_accuracy, 2),
    "validation_source": "Held-out exercise dataset",
    "note": "These metrics evaluate muscle-region classification, not Apriori recommendations."
}

info_filename = "fitroutine_classifier_info.json"

with open(info_filename, "w") as f:
    json.dump(model_info, f, indent=4)

print("Model saved:", model_filename)
print("Model information saved:", info_filename)
print("Classes:", final_region_model.classes_.tolist())

files.download(model_filename)
files.download(info_filename)


Model saved: fitroutine_muscle_region_classifier.joblib
Model information saved: fitroutine_classifier_info.json
Classes: ['Core', 'Lower Body', 'Neck', 'Upper Body']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [45]:
print("Model available:", "final_region_model" in globals())
print("Validation data available:", "X_val_region" in globals())
print("Validation labels available:", "y_val_region" in globals())

Model available: True
Validation data available: True
Validation labels available: True


In [46]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

predictions = final_region_model.predict(X_val_region)
probabilities = final_region_model.predict_proba(X_val_region)

confidence_scores = np.max(probabilities, axis=1)

thresholds = [0.40, 0.50, 0.60, 0.70, 0.80]

results = []

for threshold in thresholds:
    accepted = confidence_scores >= threshold
    accepted_count = np.sum(accepted)

    coverage = accepted_count / len(predictions)

    if accepted_count > 0:
        accepted_accuracy = accuracy_score(
            np.asarray(y_val_region)[accepted],
            predictions[accepted]
        )
    else:
        accepted_accuracy = np.nan

    results.append({
        "Threshold": threshold,
        "Accepted": accepted_count,
        "Uncertain": len(predictions) - accepted_count,
        "Coverage (%)": round(coverage * 100, 2),
        "Accepted Accuracy (%)": round(accepted_accuracy * 100, 2)
    })

results_df = pd.DataFrame(results)
display(results_df)

,Threshold,Accepted,Uncertain,Coverage (%),Accepted Accuracy (%)
0,0.4,155,20,88.57,94.84
1,0.5,128,47,73.14,96.88
2,0.6,82,93,46.86,98.78
3,0.7,47,128,26.86,97.87
4,0.8,17,158,9.71,100.00


In [47]:
threshold = 0.40

region_check = pd.DataFrame({
    "Actual Region": np.asarray(y_val_region),
    "Predicted Region": predictions,
    "Confidence": confidence_scores
})

region_check["Status"] = np.where(
    region_check["Confidence"] >= threshold,
    "Accepted",
    "Uncertain"
)

print("Predictions by actual muscle region:")
display(
    pd.crosstab(
        region_check["Actual Region"],
        region_check["Status"]
    )
)

Predictions by actual muscle region:


Status,Accepted,Uncertain
Actual Region,,
Core,18,5
Lower Body,57,3
Neck,2,0
Upper Body,78,12


In [48]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report

cv_predictions = cross_val_predict(
    region_pipeline,
    X_train_region,
    y_train_region,
    cv=cv,
    n_jobs=1
)

print("=== 5-FOLD CROSS-VALIDATION CLASSIFICATION REPORT ===")
print(
    classification_report(
        y_train_region,
        cv_predictions,
        zero_division=0
    )
)

=== 5-FOLD CROSS-VALIDATION CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

        Core       0.62      0.65      0.63        97
  Lower Body       0.83      0.87      0.85       238
        Neck       0.80      0.67      0.73         6
  Upper Body       0.92      0.88      0.90       359

    accuracy                           0.84       700
   macro avg       0.79      0.77      0.78       700
weighted avg       0.84      0.84      0.84       700



In [49]:
import pandas as pd

error_analysis = pd.DataFrame({
    "Exercise Text": X_train_region.values,
    "Actual Region": y_train_region.values,
    "Predicted Region": cv_predictions
})

core_errors = error_analysis[
    (error_analysis["Actual Region"] == "Core") &
    (error_analysis["Predicted Region"] != "Core")
]

print("=== MISCLASSIFIED CORE EXERCISES ===")
print("Total:", len(core_errors))

display(core_errors.head(15))

=== MISCLASSIFIED CORE EXERCISES ===
Total: 34


,Exercise Text,Actual Region,Predicted Region
5,Suspended Fallout Adjust the straps so the han...,Core,Upper Body
12,Hyperextensions With No Hyperextension Bench W...,Core,Upper Body
18,Advanced Kettlebell Windmill Clean and press a...,Core,Upper Body
104,Crossover Reverse Lunge Stand with your feet s...,Core,Lower Body
107,Seated Good Mornings Set up a box in a power r...,Core,Lower Body
115,Seated Barbell Twist Start out by sitting at t...,Core,Upper Body
168,Standing Pelvic Tilt Start off with your feet ...,Core,Lower Body
190,Deficit Deadlift Begin by having a platform or...,Core,Lower Body
191,Kettlebell Windmill Place a kettlebell in fron...,Core,Upper Body
202,Supine Two-Arm Overhead Throw Lay on the groun...,Core,Upper Body


In [50]:
names_to_check = [
    "Side Bridge",
    "Deficit Deadlift",
    "Suspended Fallout",
    "Seated Barbell Twist"
]

for name in names_to_check:
    matches = region_data[
        region_data["model_text"].str.contains(
            name, case=False, regex=False, na=False
        )
    ]

    print(f"\n=== {name} ===")
    display(
        matches[["muscle_region", "model_text"]].head(2)
    )


=== Side Bridge ===


,muscle_region,model_text
664,Core,Side Bridge



=== Deficit Deadlift ===


,muscle_region,model_text
205,Core,Deficit Deadlift Begin by having a platform or...



=== Suspended Fallout ===


,muscle_region,model_text
808,Core,Suspended Fallout Adjust the straps so the han...



=== Seated Barbell Twist ===


,muscle_region,model_text
623,Core,Seated Barbell Twist Start out by sitting at t...


In [51]:
text_lengths = X_train_region.fillna("").str.split().str.len()

print("=== EXERCISE TEXT LENGTH ANALYSIS ===")
print("Total training exercises:", len(text_lengths))
print("Exercises with 3 words or fewer:", (text_lengths <= 3).sum())
print("Exercises with 10 words or fewer:", (text_lengths <= 10).sum())
print("Median word count:", text_lengths.median())

print("\n=== SHORTEST EXERCISE TEXTS ===")
display(
    X_train_region.loc[
        text_lengths.nsmallest(15).index
    ].to_frame(name="Exercise Text")
)

=== EXERCISE TEXT LENGTH ANALYSIS ===
Total training exercises: 700
Exercises with 3 words or fewer: 4
Exercises with 10 words or fewer: 4
Median word count: 108.0

=== SHORTEST EXERCISE TEXTS ===


,Exercise Text
664,Side Bridge
357,Iron Cross
567,Push Press
500,One-Arm Kettlebell Swings
662,Shoulder Stretch Reach your left arm across yo...
87,"Bicycling To begin, seat yourself on the bike ..."
661,Shoulder Raise Relax your arms to your sides a...
839,Upper Back Stretch Clasp fingers together with...
843,Upward Stretch Extend both hands straight abov...
830,Triceps Stretch Reach your hand behind your he...


In [52]:
print("=== AVAILABLE DATASET COLUMNS ===")
print(region_data.columns.tolist())

print("\n=== SAMPLE EXERCISES ===")
display(region_data.head(3))

=== AVAILABLE DATASET COLUMNS ===
['name', 'force', 'level', 'mechanic', 'equipment', 'primaryMuscles', 'secondaryMuscles', 'instructions', 'category', 'images', 'id', 'primary_muscle_count', 'primary_muscle', 'instruction_text', 'model_text', 'muscle_region']

=== SAMPLE EXERCISES ===


,name,force,level,mechanic,equipment,primaryMuscles,secondaryMuscles,instructions,category,images,id,primary_muscle_count,primary_muscle,instruction_text,model_text,muscle_region
0,3/4 Sit-Up,pull,beginner,compound,body only,[abdominals],[],[Lie down on the floor and secure your feet. Y...,strength,"[3_4_Sit-Up/0.jpg, 3_4_Sit-Up/1.jpg]",3_4_Sit-Up,1,abdominals,Lie down on the floor and secure your feet. Yo...,3/4 Sit-Up Lie down on the floor and secure yo...,Core
1,90/90 Hamstring,push,beginner,None,body only,[hamstrings],[calves],"[Lie on your back, with one leg extended strai...",stretching,"[90_90_Hamstring/0.jpg, 90_90_Hamstring/1.jpg]",90_90_Hamstring,1,hamstrings,"Lie on your back, with one leg extended straig...","90/90 Hamstring Lie on your back, with one leg...",Lower Body
2,Ab Crunch Machine,pull,intermediate,isolation,machine,[abdominals],[],[Select a light resistance and sit down on the...,strength,"[Ab_Crunch_Machine/0.jpg, Ab_Crunch_Machine/1....",Ab_Crunch_Machine,1,abdominals,Select a light resistance and sit down on the ...,Ab Crunch Machine Select a light resistance an...,Core


In [53]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

# Use only the original training exercises
train_rows = region_data.loc[X_train_muscle.index].copy()

# Repeat the exercise name to give it more weight
X_name_weighted = (
    train_rows["name"].fillna("") + " " +
    train_rows["name"].fillna("") + " " +
    train_rows["name"].fillna("") + " " +
    train_rows["instruction_text"].fillna("")
)

y_name_weighted = train_rows["muscle_region"]

weighted_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 1),
        min_df=5,
        max_features=300
    )),
    ("classifier", LogisticRegression(
        C=0.5,
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    weighted_pipeline,
    X_name_weighted,
    y_name_weighted,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy"
    },
    return_train_score=True,
    n_jobs=1
)

train_acc = scores["train_accuracy"].mean() * 100
val_acc = scores["test_accuracy"].mean() * 100
gap = abs(train_acc - val_acc)
balanced_acc = scores["test_balanced_accuracy"].mean() * 100

print("=== NAME-WEIGHTED CLASSIFIER ===")
print(f"CV training accuracy: {train_acc:.2f}%")
print(f"CV validation accuracy: {val_acc:.2f}%")
print(f"CV accuracy gap: {gap:.2f} percentage points")
print(f"CV balanced accuracy: {balanced_acc:.2f}%")

print("\n=== PROFESSOR'S REQUIREMENTS ===")
print("CV validation accuracy >= 85%:",
      "PASS" if val_acc >= 85 else "NOT YET")
print("CV accuracy gap <= 5 percentage points:",
      "PASS" if gap <= 5 else "NOT YET")

=== NAME-WEIGHTED CLASSIFIER ===
CV training accuracy: 91.18%
CV validation accuracy: 83.43%
CV accuracy gap: 7.75 percentage points
CV balanced accuracy: 76.56%

=== PROFESSOR'S REQUIREMENTS ===
CV validation accuracy >= 85%: NOT YET
CV accuracy gap <= 5 percentage points: NOT YET


In [54]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

test_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 1),
        min_df=5,
        max_features=300
    )),
    ("classifier", LogisticRegression(
        C=0.4,
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    test_pipeline,
    X_train_region,
    y_train_region,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy"
    },
    return_train_score=True,
    n_jobs=1
)

train_acc = scores["train_accuracy"].mean() * 100
val_acc = scores["test_accuracy"].mean() * 100
gap = abs(train_acc - val_acc)

print("=== C=0.4 EXPERIMENT ===")
print(f"CV training accuracy: {train_acc:.2f}%")
print(f"CV validation accuracy: {val_acc:.2f}%")
print(f"CV accuracy gap: {gap:.2f} percentage points")
print(f"CV balanced accuracy: {scores['test_balanced_accuracy'].mean() * 100:.2f}%")

=== C=0.4 EXPERIMENT ===
CV training accuracy: 88.82%
CV validation accuracy: 83.43%
CV accuracy gap: 5.39 percentage points
CV balanced accuracy: 77.45%


In [55]:
train_data = region_data.loc[X_train_muscle.index].copy()

print("=== DUPLICATE EXERCISE NAME CHECK ===")
print("Total training exercises:", len(train_data))
print(
    "Duplicate exercise names:",
    train_data["name"].duplicated().sum()
)

duplicates = train_data[
    train_data["name"].duplicated(keep=False)
].sort_values("name")

display(
    duplicates[
        ["name", "primary_muscle", "muscle_region"]
    ].head(20)
)

=== DUPLICATE EXERCISE NAME CHECK ===
Total training exercises: 700
Duplicate exercise names: 0


,name,primary_muscle,muscle_region


In [56]:

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

# Use the same original 700 training exercises
X_names = region_data.loc[X_train_muscle.index, "name"].fillna("")
y_names = region_data.loc[X_train_muscle.index, "muscle_region"]

name_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2
    )),
    ("classifier", LogisticRegression(
        C=0.5,
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    name_pipeline,
    X_names,
    y_names,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy"
    },
    return_train_score=True,
    n_jobs=1
)

train_acc = scores["train_accuracy"].mean() * 100
val_acc = scores["test_accuracy"].mean() * 100
gap = abs(train_acc - val_acc)

print("=== EXERCISE NAME-ONLY EXPERIMENT ===")
print(f"CV training accuracy: {train_acc:.2f}%")
print(f"CV validation accuracy: {val_acc:.2f}%")
print(f"CV accuracy gap: {gap:.2f} percentage points")
print(f"CV balanced accuracy: {scores['test_balanced_accuracy'].mean() * 100:.2f}%")


=== EXERCISE NAME-ONLY EXPERIMENT ===
CV training accuracy: 89.50%
CV validation accuracy: 78.14%
CV accuracy gap: 11.36 percentage points
CV balanced accuracy: 73.13%


In [57]:

# Use the same 700 training exercises
X_instructions = region_data.loc[
    X_train_muscle.index, "instruction_text"
].fillna("")

y_instructions = region_data.loc[
    X_train_muscle.index, "muscle_region"
]

instruction_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 1),
        min_df=5,
        max_features=300
    )),
    ("classifier", LogisticRegression(
        C=0.5,
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ))
])

scores = cross_validate(
    instruction_pipeline,
    X_instructions,
    y_instructions,
    cv=StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    ),
    scoring={
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy"
    },
    return_train_score=True,
    n_jobs=1
)

train_acc = scores["train_accuracy"].mean() * 100
val_acc = scores["test_accuracy"].mean() * 100
gap = abs(train_acc - val_acc)

print("=== INSTRUCTIONS-ONLY EXPERIMENT ===")
print(f"CV training accuracy: {train_acc:.2f}%")
print(f"CV validation accuracy: {val_acc:.2f}%")
print(f"CV accuracy gap: {gap:.2f} percentage points")
print(f"CV balanced accuracy: {scores['test_balanced_accuracy'].mean() * 100:.2f}%")


=== INSTRUCTIONS-ONLY EXPERIMENT ===
CV training accuracy: 89.14%
CV validation accuracy: 81.71%
CV accuracy gap: 7.43 percentage points
CV balanced accuracy: 75.55%


In [58]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 1),
    min_df=5,
    max_features=300
)

vectorizer.fit(X_train_region)

print("=== TF-IDF VOCABULARY CHECK ===")
print("Number of features:", len(vectorizer.get_feature_names_out()))
print("Sample features:", vectorizer.get_feature_names_out()[:30])

=== TF-IDF VOCABULARY CHECK ===
Number of features: 300
Sample features: ['10' '20' '30' '90' 'adjust' 'allow' 'allowing' 'alternating' 'angle'
 'ankles' 'apart' 'appropriate' 'arm' 'arms' 'attach' 'attachment'
 'attempt' 'avoid' 'away' 'balance' 'ball' 'band' 'bands' 'bar' 'barbell'
 'begin' 'bench' 'bend' 'bending' 'bent']


In [59]:

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for feature_limit in [100, 200, 300, 500, 800, 1200]:
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 1),
            min_df=5,
            max_features=feature_limit
        )),
        ("classifier", LogisticRegression(
            C=0.5,
            max_iter=3000,
            class_weight="balanced",
            random_state=42
        ))
    ])

    scores = cross_validate(
        pipeline,
        X_train_region,
        y_train_region,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy"
        },
        return_train_score=True,
        n_jobs=1
    )

    train_acc = scores["train_accuracy"].mean() * 100
    val_acc = scores["test_accuracy"].mean() * 100
    gap = abs(train_acc - val_acc)

    results.append({
        "Max Features": feature_limit,
        "CV Training (%)": round(train_acc, 2),
        "CV Validation (%)": round(val_acc, 2),
        "Gap (pp)": round(gap, 2),
        "Balanced Accuracy (%)": round(
            scores["test_balanced_accuracy"].mean() * 100, 2
        ),
        "Meets Both": "YES" if val_acc >= 85 and gap <= 5 else "NO"
    })

display(pd.DataFrame(results))


,Max Features,CV Training (%),CV Validation (%),Gap (pp),Balanced Accuracy (%),Meets Both
0,100,85.07,79.29,5.79,77.82,NO
1,200,87.93,82.57,5.36,76.27,NO
2,300,89.43,84.14,5.29,77.99,NO
3,500,90.71,84.00,6.71,77.53,NO
4,800,91.32,84.43,6.89,77.81,NO
5,1200,91.32,84.43,6.89,77.81,NO


In [60]:

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_validate

svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 1),
        min_df=5,
        max_features=300
    )),
    ("classifier", LinearSVC(
        C=0.5,
        class_weight="balanced",
        random_state=42,
        max_iter=5000
    ))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    svm_pipeline,
    X_train_region,
    y_train_region,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy"
    },
    return_train_score=True,
    n_jobs=1
)

train_acc = scores["train_accuracy"].mean() * 100
val_acc = scores["test_accuracy"].mean() * 100
gap = abs(train_acc - val_acc)
balanced_acc = scores["test_balanced_accuracy"].mean() * 100

print("=== LINEAR SVM EXPERIMENT ===")
print(f"CV training accuracy: {train_acc:.2f}%")
print(f"CV validation accuracy: {val_acc:.2f}%")
print(f"CV accuracy gap: {gap:.2f} percentage points")
print(f"CV balanced accuracy: {balanced_acc:.2f}%")

print("\n=== PROFESSOR'S REQUIREMENTS ===")
print("CV validation accuracy >= 85%:",
      "PASS" if val_acc >= 85 else "NOT YET")
print("CV accuracy gap <= 5 percentage points:",
      "PASS" if gap <= 5 else "NOT YET")


=== LINEAR SVM EXPERIMENT ===
CV training accuracy: 95.82%
CV validation accuracy: 85.14%
CV accuracy gap: 10.68 percentage points
CV balanced accuracy: 77.10%

=== PROFESSOR'S REQUIREMENTS ===
CV validation accuracy >= 85%: PASS
CV accuracy gap <= 5 percentage points: NOT YET


In [61]:

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for c_value in [0.01, 0.03, 0.1, 0.2, 0.3, 0.5]:
    svm_pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 1),
            min_df=5,
            max_features=300
        )),
        ("classifier", LinearSVC(
            C=c_value,
            class_weight="balanced",
            random_state=42,
            max_iter=5000
        ))
    ])

    scores = cross_validate(
        svm_pipeline,
        X_train_region,
        y_train_region,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy"
        },
        return_train_score=True,
        n_jobs=1
    )

    train_acc = scores["train_accuracy"].mean() * 100
    val_acc = scores["test_accuracy"].mean() * 100
    gap = abs(train_acc - val_acc)

    results.append({
        "C": c_value,
        "CV Training (%)": round(train_acc, 2),
        "CV Validation (%)": round(val_acc, 2),
        "Gap (pp)": round(gap, 2),
        "Balanced Accuracy (%)": round(
            scores["test_balanced_accuracy"].mean() * 100, 2
        ),
        "Meets Both": "YES" if val_acc >= 85 and gap <= 5 else "NO"
    })

print("=== LINEAR SVM REGULARIZATION COMPARISON ===")
display(pd.DataFrame(results))


=== LINEAR SVM REGULARIZATION COMPARISON ===


,C,CV Training (%),CV Validation (%),Gap (pp),Balanced Accuracy (%),Meets Both
0,0.01,81.43,78.43,3.00,63.45,NO
1,0.03,86.61,82.29,4.32,70.83,NO
2,0.10,90.86,83.29,7.57,73.73,NO
3,0.20,93.14,84.71,8.43,76.30,NO
4,0.30,94.11,84.71,9.39,76.65,NO
5,0.50,95.82,85.14,10.68,77.10,NO


In [62]:
from sklearn.metrics import confusion_matrix
import pandas as pd
labels = ["Core", "Lower Body", "Neck", "Upper Body"]
cm = confusion_matrix(y_val_region, val_predictions_region, labels=labels)
display(pd.DataFrame(cm, index=labels, columns=labels))

,Core,Lower Body,Neck,Upper Body
Core,18,3,1,1
Lower Body,4,56,0,0
Neck,0,0,2,0
Upper Body,6,1,0,83
